# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata and display title and description
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs using the Croissant metadata loaded from the dataset.

We will list all `@id`s for RecordSets and, for the main RecordSet, its Fields (and their `@id`s), for clarity and to aid subsequent referencing.

In [ ]:
# List all available RecordSets and their fields, referencing entities by `@id`
record_sets_info = []
for recordset in metadata.record_sets:
    fields_list = []
    for field in recordset.fields:
        field_info = {
            'field_name': field.name,
            'field_id': field.id,
            'dataType': getattr(field, 'data_type', None)
        }
        # list column ids if present
        if hasattr(field, 'columns') and field.columns:
            field_info['columns'] = [c.id for c in field.columns if hasattr(c, 'id')]
        fields_list.append(field_info)
    record_sets_info.append({'recordset_name': recordset.name, 'recordset_id': recordset.id, 'fields': fields_list})

print(f"Record Sets in this dataset (refer to @id fields):")
for rs in record_sets_info:
    print(f"- RecordSet Name: {rs['recordset_name']}\n  @id: {rs['recordset_id']}")
    print("  Fields (by @id):")
    for field in rs['fields']:
        print(f"    - {field['field_name']}  (@id: {field['field_id']})  DataType: {field.get('dataType', None)}")
        if 'columns' in field:
            print(f"      Columns (by @id): {field['columns']}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview. Here, we extract all main tabular RecordSets.

In [ ]:
# Extract data from each RecordSet into a DataFrame - using their `@id`.

# Collect all recordset @id's
record_set_ids = [rs['recordset_id'] for rs in record_sets_info]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading RecordSet: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
        else:
            print(f"Warning: RecordSet '{record_set_id}' yields no records!")
    except Exception as e:
        print(f"Error loading {record_set_id}: {e}")

if not dataframes:
    print("No record sets could be loaded into DataFrames.")
else:
    # Show the columns for the first loaded DataFrame
    example_rs = list(dataframes.keys())[0]
    print(f"Loaded DataFrame from RecordSet '@id': {example_rs}")
    print('Columns:', dataframes[example_rs].columns.tolist())
    display(dataframes[example_rs].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section demonstrates removing outliers, transforming data, and grouping by key attributes, referencing fields via their unique `@id` as per Croissant.

Please substitute the field `@id`s with the appropriate values from Section 2 if your field choices differ.

In [ ]:
# Example EDA: filter, normalize, group using field `@id`s

if not dataframes:
    print('No DataFrames loaded in previous step. Please check dataset RecordSets.')
else:
    # Pick the main table for further EDA
    main_rs_id = list(dataframes.keys())[0]
    df = dataframes[main_rs_id]
    print(f'Working with RecordSet: {main_rs_id}')

    # Try to find a numeric field automatically
    numeric_cols = df.select_dtypes(include='number').columns.tolist()
    if numeric_cols:
        numeric_field_id = numeric_cols[0]
    else:
        print('No numeric columns found.')
        numeric_field_id = None

    if numeric_field_id is not None:
        threshold = df[numeric_field_id].mean()  # Example threshold: mean
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records where '{numeric_field_id}' > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize the field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized '{numeric_field_id}' for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by another field (choose first available object-type field as example)
        candidate_group_fields = df.select_dtypes(include='object').columns.tolist()
        group_field_id = candidate_group_fields[0] if candidate_group_fields else None
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped data by '{group_field_id}':")
            display(grouped_df.head())
        else:
            print('No suitable categorical field for grouping.')
    else:
        print('Please select a valid numeric field for analysis.')

## 5. Visualization
Visualize key data distributions or relationships. Example below: distribution/histogram of a numeric field, and a bar plot of group means (if grouping succeeded).

In [ ]:
import matplotlib.pyplot as plt

if not dataframes:
    print('No DataFrames loaded to visualize.')
else:
    df = dataframes[list(dataframes.keys())[0]]
    # Numeric field found in previous step?
    if 'numeric_field_id' in locals() and numeric_field_id in df.columns:
        plt.figure(figsize=(8,4))
        df[numeric_field_id].hist(bins=30, color='skyblue', edgecolor='black')
        plt.title(f"Distribution of '{numeric_field_id}'")
        plt.xlabel(numeric_field_id)
        plt.ylabel('Frequency')
        plt.show()

        # Barplot for group means
        if 'group_field_id' in locals() and group_field_id in df.columns:
            group_means = df.groupby(group_field_id)[numeric_field_id].mean().sort_values()
            plt.figure(figsize=(10,4))
            group_means.plot(kind='bar')
            plt.title(f"Mean {numeric_field_id} by {group_field_id}")
            plt.xlabel(group_field_id)
            plt.ylabel(f"Mean {numeric_field_id}")
            plt.tight_layout()
            plt.show()
        else:
            print('No group field available for bar plot.')
    else:
        print('No numeric field found for histogram.')

## 6. Conclusion
In this notebook, we have used the `mlcroissant` library to load the FAIR^2 Croissant dataset on mass spectrometry analysis of extracellular vesicles. We explored its schema using the Croissant `@id` mechanisms, loaded records by record set, performed basic exploratory analysis, and visualized data distributions. This workflow can easily be extended to other Croissant-compliant datasets simply by updating the schema URL and referencing the new record set and field IDs.